In [4]:
import pandas as pd
import numpy as np
import random
from pathlib import Path

# Reproducible random data for the exercise
random.seed(42)
np.random.seed(42)

n = 30
names = ["Alice", "BOB", " Charlie ", "diana", "Ethan", "Fiona", "george", "Hannah"]
departments = ["Finance", "IT", "HR", "Sales", " finance ", "it", None]
cities = ["Phnom Penh", "Siem Reap", "Battambang", "phnom penh", None]

rows = []
for i in range(1, n + 1):
    name = random.choice(names)
    age = random.randint(18, 65)
    salary = random.randint(250, 2200)
    department = random.choice(departments)
    city = random.choice(cities)

    # Introduce intentional data-quality problems
    if random.random() < 0.13:
        age = None
    if random.random() < 0.10:
        salary = None
    if random.random() < 0.12:
        name = None
    if random.random() < 0.08:
        age = random.choice([-5, 150])  # invalid age
    if random.random() < 0.08:
        salary = random.choice([-100, 999999])  # invalid salary

    rows.append({
        "employee_id": i,
        "name": name,
        "age": age,
        "department": department,
        "salary_usd": salary,
        "city": city
    })

# Add duplicates deliberately
rows.append(rows[4].copy())
rows.append(rows[11].copy())

df = pd.DataFrame(rows)
path = Path("random_employee_data.csv")
df.to_csv(path, index=False)

print(f"Created CSV: {path}")
print(f"Rows: {len(df)}")
print("\nFirst 10 rows:")
print(df.head(10).to_string(index=False))
print("\nMissing values per column:")
print(df.isna().sum().to_string())


Created CSV: random_employee_data.csv
Rows: 32

First 10 rows:
 employee_id      name  age department  salary_usd       city
           1       BOB 19.0         HR      1768.0  Siem Reap
           2    george 20.0    Finance       311.0  Siem Reap
           3    george 32.0   finance          NaN Battambang
           4  Charlie   NaN         HR         NaN Phnom Penh
           5    Hannah 52.0      Sales       505.0 Phnom Penh
           6       BOB 20.0         IT      1604.0 Battambang
           7     Fiona -5.0         HR      1008.0  Siem Reap
           8     diana 28.0      Sales      1196.0 Battambang
           9     Alice 32.0    Finance         NaN Battambang
          10     diana 59.0      Sales      1272.0 phnom penh

Missing values per column:
employee_id    0
name           3
age            1
department     5
salary_usd     3
city           8


In [5]:
import time
from functools import wraps

def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[START] {func.__name__}")
        try:
            return func(*args, **kwargs)
        finally:
            print(f"[FINISH] {func.__name__}")
    return wrapper


def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIME] {func.__name__} took {elapsed:.4f} seconds")
        return result
    return wrapper

In [6]:
class DataCleaner:

    @log_action
    @log_time
    def clean(self, df):
        df = df.copy()

        # Remove duplicate rows
        df = df.drop_duplicates()

        # Standardize text
        for col in ["name", "department", "city"]:
            df[col] = df[col].astype("string").str.strip()

        df["name"] = df["name"].str.title()
        df["department"] = df["department"].str.title()
        df["city"] = df["city"].str.title()

        # Replace invalid values with missing values
        df.loc[~df["age"].between(18, 100), "age"] = pd.NA
        df.loc[df["salary_usd"] < 0, "salary_usd"] = pd.NA
        df.loc[df["salary_usd"] > 100_000, "salary_usd"] = pd.NA

        # Fill missing values
        for col in ["name", "department", "city"]:
            df[col] = df[col].fillna("Unknown")

        df["age"] = df["age"].fillna(df["age"].median())
        df["salary_usd"] = df["salary_usd"].fillna(
            df["salary_usd"].median()
        )

        return df

In [7]:
cleaner = DataCleaner()

cleaned_df = cleaner.clean(df)

cleaned_df.head()

[START] clean
[TIME] clean took 0.0133 seconds
[FINISH] clean


,employee_id,name,age,department,salary_usd,city
0,1,Bob,19.0,Hr,1768.0,Siem Reap
1,2,George,20.0,Finance,311.0,Siem Reap
2,3,George,32.0,Finance,1196.0,Battambang
3,4,Charlie,42.0,Hr,1196.0,Phnom Penh
4,5,Hannah,52.0,Sales,505.0,Phnom Penh
